# 06. Download European data
Hourly prices for 26 bidding zones and solar, wind and consumption for the 15 largest, since 2019. The raw files stay on my computer (data/raw/entsoe is in .gitignore).

In [1]:
# Import the libraries, connect to the ENTSO-E API and create the folder for the raw files
import os
import pandas as pd
from entsoe import EntsoePandasClient

token = open("../entsoe_token.txt").read().strip()
client = EntsoePandasClient(api_key=token)
os.makedirs("../data/raw/entsoe", exist_ok=True)

In [2]:
# Bidding zones to download (one per country) and the period
zones = {
    "PT": "Portugal", "ES": "Spain", "FR": "France", "BE": "Belgium", "NL": "Netherlands",
    "DE_LU": "Germany", "AT": "Austria", "CH": "Switzerland", "IT_NORD": "Italy (North)",
    "PL": "Poland", "CZ": "Czechia", "SK": "Slovakia", "HU": "Hungary", "SI": "Slovenia",
    "HR": "Croatia", "RO": "Romania", "BG": "Bulgaria", "GR": "Greece", "DK_1": "Denmark (West)",
    "SE_3": "Sweden (Stockholm)", "NO_1": "Norway (Oslo)", "FI": "Finland", "EE": "Estonia",
    "LV": "Latvia", "LT": "Lithuania", "IE_SEM": "Ireland",
}
start = pd.Timestamp("2019-01-01", tz="Europe/Brussels")
today = pd.Timestamp.today(tz="Europe/Brussels").normalize()
end_prices = today + pd.Timedelta(days=2)
end_generation = today

In [3]:
# Download the hourly prices of each zone since 2019 (zones already downloaded are skipped)
for zone in zones:
    file = "../data/raw/entsoe/prices_" + zone + ".csv"
    if os.path.exists(file):
        print(zone, "already downloaded")
        continue
    try:
        prices = client.query_day_ahead_prices(zone, start=start, end=end_prices)
        prices = prices.resample("h").mean().tz_convert("UTC")
        prices.to_csv(file, index_label="time_utc", header=["price"])
        print(zone, len(prices), "hours")
    except Exception as error:
        print(zone, "failed:", error)

PT 68111 hours
ES 68111 hours
FR 68087 hours
BE 68111 hours
NL 68087 hours
DE_LU 68087 hours
AT 68111 hours
CH 68111 hours
IT_NORD 68087 hours
PL 68111 hours
CZ 68111 hours
SK 68111 hours
HU 68111 hours
SI 68111 hours
HR 68111 hours
RO 68111 hours
BG 68111 hours
GR 68111 hours
DK_1 68111 hours
SE_3 68111 hours
NO_1 68111 hours
FI 68111 hours
EE 68111 hours
LV 68111 hours
LT 68111 hours
IE_SEM 67895 hours


In [4]:
# Check: average price in 2025 in each zone
for zone in zones:
    file = "../data/raw/entsoe/prices_" + zone + ".csv"
    if os.path.exists(file):
        data = pd.read_csv(file)
        year_2025 = data[data["time_utc"].str[:4] == "2025"]
        print(zones[zone], round(year_2025["price"].mean(), 1), "€/MWh")

Portugal 66.2 €/MWh
Spain 65.3 €/MWh
France 61.1 €/MWh
Belgium 82.6 €/MWh
Netherlands 86.8 €/MWh
Germany 89.3 €/MWh
Austria 98.9 €/MWh
Switzerland 101.7 €/MWh
Italy (North) 115.9 €/MWh
Poland 104.3 €/MWh
Czechia 96.8 €/MWh
Slovakia 102.7 €/MWh
Hungary 108.5 €/MWh
Slovenia 104.6 €/MWh
Croatia 105.4 €/MWh
Romania 108.2 €/MWh
Bulgaria 106.9 €/MWh
Greece 103.6 €/MWh
Denmark (West) 81.1 €/MWh
Sweden (Stockholm) 46.2 €/MWh
Norway (Oslo) 58.4 €/MWh
Finland 40.5 €/MWh
Estonia 80.4 €/MWh
Latvia 85.7 €/MWh
Lithuania 85.3 €/MWh
Ireland 114.4 €/MWh


In [3]:
# Download solar, wind and consumption of one zone for one year, as hourly values in UTC
def download_year(zone, year):
    year_start = pd.Timestamp(str(year) + "-01-01", tz="Europe/Brussels")
    year_end = min(pd.Timestamp(str(year + 1) + "-01-01", tz="Europe/Brussels"), end_generation)

    generation = client.query_generation(zone, start=year_start, end=year_end)
    if isinstance(generation.columns, pd.MultiIndex):
        generation = generation.xs("Actual Aggregated", axis=1, level=1)
    generation = generation.resample("h").mean()
    load = client.query_load(zone, start=year_start, end=year_end).resample("h").mean()

    table = pd.DataFrame(index=generation.index)
    table["solar"] = 0
    table["wind"] = 0
    if "Solar" in generation.columns:
        table["solar"] = generation["Solar"]
    if "Wind Onshore" in generation.columns:
        table["wind"] = table["wind"] + generation["Wind Onshore"].fillna(0)
    if "Wind Offshore" in generation.columns:
        table["wind"] = table["wind"] + generation["Wind Offshore"].fillna(0)
    table["load"] = load["Actual Load"]
    return table.tz_convert("UTC")

In [4]:
# Main countries for the production data
production_zones = ["PT", "ES", "FR", "DE_LU", "IT_NORD", "NL", "BE", "PL", "AT",
                    "DK_1", "FI", "GR", "HU", "RO", "CZ"]

# Download year by year and save one file per zone (zones already downloaded are skipped)
for zone in production_zones:
    file = "../data/raw/entsoe/production_" + zone + ".csv"
    if os.path.exists(file):
        print(zone, "already downloaded")
        continue
    years = []
    for year in range(2019, today.year + 1):
        try:
            years.append(download_year(zone, year))
            print("  ", zone, year, "done")
        except Exception as error:
            print("  ", zone, year, "failed:", error)
    if len(years) > 0:
        table = pd.concat(years)
        table.to_csv(file, index_label="time_utc")
        print(zone, len(table), "hours")

   PT 2019 done
   PT 2020 done
   PT 2021 done
   PT 2022 done
   PT 2023 done
   PT 2024 done
   PT 2025 done
   PT 2026 done
PT 68063 hours
   ES 2019 done
   ES 2020 done
   ES 2021 done
   ES 2022 done
   ES 2023 done
   ES 2024 done
   ES 2025 done
   ES 2026 done
ES 68063 hours
   FR 2019 done
   FR 2020 done
   FR 2021 done
   FR 2022 done
   FR 2023 done
   FR 2024 done
   FR 2025 done
   FR 2026 done
FR 68063 hours
   DE_LU 2019 done
   DE_LU 2020 done
   DE_LU 2021 done
   DE_LU 2022 done
   DE_LU 2023 done
   DE_LU 2024 done
   DE_LU 2025 done
   DE_LU 2026 done
DE_LU 68063 hours
   IT_NORD 2019 done
   IT_NORD 2020 done
   IT_NORD 2021 done
   IT_NORD 2022 done
   IT_NORD 2023 done
   IT_NORD 2024 done
   IT_NORD 2025 done
   IT_NORD 2026 done
IT_NORD 68063 hours
   NL 2019 done
   NL 2020 done
   NL 2021 done
   NL 2022 done
   NL 2023 done
   NL 2024 done
   NL 2025 done
   NL 2026 done
NL 68063 hours
   BE 2019 done
   BE 2020 done
   BE 2021 done
   BE 2022 done
   BE 

In [5]:
# Check the production files: number of hours, first and last hour, and average solar, wind and consumption
for zone in production_zones:
    table = pd.read_csv("../data/raw/entsoe/production_" + zone + ".csv")
    print(zone, len(table), "hours", table["time_utc"].min(), "to", table["time_utc"].max(),
          "| solar", round(table["solar"].mean()), "| wind", round(table["wind"].mean()),
          "| load", round(table["load"].mean()))

PT 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 393 | wind 1490 | load 5809
ES 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 4175 | wind 6439 | load 27272
FR 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 2377 | wind 4709 | load 50510
DE_LU 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 6784 | wind 14832 | load 55120
IT_NORD 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 1046 | wind 31 | load 18040
NL 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 46 | wind 1841 | load 12609
BE 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 808 | wind 1303 | load 9334
PL 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 1311 | wind 2160 | load 18980
AT 68063 hours 2018-12-31 23:00:00+00:00 to 2026-10-06 21:00:00+00:00 | solar 342 | wind 911 | load 6894
DK_1 68063 hours 2018-12-31 23